# 05 Customer Segmentation


In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    markers = ("src", "data", "notebooks")

    for p in [start, *start.parents]:
        if p.name == "customer-segmentation-analytics" and all((p / m).is_dir() for m in markers):
            return p

    for p in [start, *start.parents]:
        candidate = p / "Improvements" / "Statistics" / "customer-segmentation-analytics"
        if all((candidate / m).is_dir() for m in markers):
            return candidate

    raise RuntimeError("Could not locate customer-segmentation-analytics project root.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")


## Objective


Build unsupervised customer clusters and compare them against provided segment labels.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler

from src.modelling import evaluate_kmeans_range, fit_kmeans

sns.set_theme(style="whitegrid")
features = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "customer_features.parquet")


## Select Segmentation Inputs


In [ ]:
seg_cols = [
    "annual_income",
    "avg_monthly_spend",
    "purchase_frequency",
    "avg_order_value",
    "return_rate",
    "engagement_score",
]

X = features[seg_cols].copy()
X_scaled = StandardScaler().fit_transform(X)


## Choose K Using Silhouette Score


In [ ]:
k_scores = evaluate_kmeans_range(X_scaled, k_values=range(2, 9), random_state=42)
k_scores


In [ ]:
plt.figure(figsize=(8, 4))
sns.lineplot(data=k_scores.sort_values("k"), x="k", y="silhouette_score", marker="o")
plt.title("Silhouette Score by K")
plt.tight_layout()
plt.show()


## Fit Final KMeans Model


In [ ]:
best_k = int(k_scores.iloc[0]["k"])
kmeans = fit_kmeans(X_scaled, n_clusters=best_k, random_state=42)
features["cluster"] = kmeans.labels_.astype(int)
features["cluster"].value_counts().sort_index()


## Cluster Profiles


In [ ]:
cluster_profile = features.groupby("cluster")[seg_cols].mean().round(2)
cluster_profile


## Cluster vs Provided Segment


In [ ]:
cluster_vs_segment = pd.crosstab(features["cluster"], features["customer_segment"], normalize="index").round(3)
cluster_vs_segment


## Save Outputs


In [ ]:
out_table = PROJECT_ROOT / "outputs" / "tables" / "customer_clusters.csv"
out_table.parent.mkdir(parents=True, exist_ok=True)
features.to_csv(out_table, index=False)
out_table
